In [1]:
import os
import glob
import sys


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb  # Has to be imported first to avoid conflicts with PyROOT
import json
from time import time
from datetime import timedelta
from yaml import safe_load, YAMLError, dump
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_curve, auc
from xgboost import XGBClassifier
from sklearn.metrics import log_loss


import config as cfg 
import efficiency_finder
#import variable_plotter as vp
plt.style.use('fcc.mplstyle')

In [2]:

#multivariate map
labels = {'signal':2,'heavy_background':1 ,'light_background':0}
labels_dict_inverted = {v: k for k, v in labels.items()}
colors = {'signal':'mediumblue','heavy_background':'y' ,'light_background':'g','background':'r'}
blobs = {'signal':'bo','heavy_background':'y+' ,'light_background':'g+','background':'r+' }


# Function to load the BDT model from a JSON file
def load_bdt_model(json_path):
    bdt_model = xgb.Booster()
    bdt_model.load_model(json_path)
    return bdt_model

# Function to load the BDT model from a JSON file
def load_bdt_model_sklearn(json_path):
    bdt_model = XGBClassifier()
    bdt_model.load_model(json_path)
    return bdt_model

# Return list of variables to use in the bdt as a python list
def vars_fromyaml(path, bdtlist):
    with open(path) as stream:
        try:
            file = safe_load(stream)
            bdtvars = file[bdtlist]
        except YAMLError as exc:
            print(exc)
    return bdtvars



In [7]:
#######################################################
# Load BDT and apply to loaded data - define as funtion
#######################################################

def load_bdt_and_apply(pickled_df_fname = "bdt_lh_dataframe.pkl", 
                        config_bdtopts = cfg.bdt_lh_opts,
                        training_round = "test_hpopt_small_sample/optimum_hps",
                        hps_dict_name = "default-hps",
                        features_list_name = "bdtlh-vars-v1",
                        bdt_label = '_lh'
                        ): # hps dict and features_list_name specift BDT used
    
    
    #path to data and outputs
    outputpath   = config_bdtopts['outputPath']
    yamlpath = cfg.fccana_opts['yamlPath']

    #Getting BDT vars for training from yaml
    bdtvars      = vars_fromyaml(yamlpath, features_list_name)
    bdtname      = f'BDT{bdt_label}_{hps_dict_name}_{features_list_name}'

    #path to pickled data
    pickled_df_path = os.path.join(outputpath, pickled_df_fname)

    #path to saved bdt
    bdt_json_path = os.path.join(outputpath,training_round,f"{bdtname}.json")

    # Load the BDT model
    try:
        print("Loading BDT model...")
        bdt_model = load_bdt_model_sklearn(bdt_json_path)
        print("BDT model loaded successfully.")
    except Exception as e:
        print(f"Error loading BDT model: {e}")
        quit()

    # Read df saved to pickle and add BDT 

    #Getting BDT vars for training from yaml
    # load pickled df
    df = pd.read_pickle(pickled_df_path)

    #add bdt score
    class_names = bdt_model.classes_
    probabilities = bdt_model.predict_proba(df[bdtvars])
    for i in range(probabilities.shape[1]):
        df[f'bdt_score_{class_names[i]}'] = probabilities[:, i]

    #calculate logloss
    validation_df = df[df["sample"]==2]
    test_df = df[df["sample"]==1]

    y_true_valid = validation_df['label']
    y_true_test = test_df['label']
    
    # Corresponding predicted probabilities (softmax output)
    y_pred_valid = validation_df[['bdt_score_0','bdt_score_1','bdt_score_2']]
    y_pred_test = test_df[['bdt_score_0','bdt_score_1','bdt_score_2']]
    
    # Compute log loss
    validation_loss = log_loss(y_true_valid, y_pred_valid)
    test_loss = log_loss(y_true_test, y_pred_test)

    print(f'validation logloss: {validation_loss}' )
    print(f'test logloss: {test_loss}' )
                            
    return bdt_model, bdtname, df


In [8]:
load_bdt_and_apply()


Loading BDT model...
BDT model loaded successfully.
validation logloss: 0.14829087952622308
test logloss: 0.14884900637106682


(XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
               colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
               early_stopping_rounds=None, enable_categorical=False,
               eval_metric='mlogloss', gamma=1.2084995905144988, gpu_id=-1,
               grow_policy='depthwise', importance_type=None,
               interaction_constraints='', learning_rate=0.19693924973969526,
               max_bin=256, max_cat_to_onehot=4, max_delta_step=0, max_depth=8,
               max_leaves=0, min_child_weight=1, missing=nan,
               monotone_constraints='()', n_estimators=493, n_jobs=0,
               num_parallel_tree=1, objective='multi:softprob', predictor='auto',
               random_state=0, reg_alpha=7.565153585635452, ...),
 'BDT_lh_default-hps_bdtlh-vars-v1',
          EVT_hemisEmin_e  EVT_hemisEmin_eCharged  EVT_hemisEmin_eNeutral  \
 0              23.707678                2.333321               21.374357   
 1              25.8799